In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Layer, Dense, Dropout, Input, Flatten, LSTM, GRU, Conv1D, MaxPooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, f1_score, accuracy_score,
                             matthews_corrcoef, confusion_matrix, roc_auc_score, average_precision_score)
from imblearn.over_sampling import RandomOverSampler
import time, warnings, os, random
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.keras.utils.set_random_seed(seed)

set_seed(42)
print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


In [2]:
# Change the path to your actual file
df = pd.read_csv(r"C:\Users\nuzha\IDD_project\IDD_DS\unsw v2\data\NF-UNSW-NB15-v2.csv")   # or NF-UNSW-NB15-v3.csv

df = df.fillna(0)
df.replace([np.inf, -np.inf], 0, inplace=True)

# Feature Engineering
df['IN_BYTE_PER_PKT']  = df['IN_BYTES']  / (df['IN_PKTS']  + 1e-6)
df['OUT_BYTE_PER_PKT'] = df['OUT_BYTES'] / (df['OUT_PKTS'] + 1e-6)
df['IN_PKT_RATE']      = df['IN_PKTS']   / (df['FLOW_DURATION_MILLISECONDS'] + 1e-6)
df['OUT_PKT_RATE']     = df['OUT_PKTS']  / (df['FLOW_DURATION_MILLISECONDS'] + 1e-6)
df['TOTAL_PKTS']       = df['IN_PKTS']   + df['OUT_PKTS']
df['TOTAL_BYTES']      = df['IN_BYTES']  + df['OUT_BYTES']
df['FLOW_DUR_PER_PKT'] = df['FLOW_DURATION_MILLISECONDS'] / (df['TOTAL_PKTS'] + 1e-6)

# TCP Flag features
df['TCP_SYN'] = df['TCP_FLAGS'].apply(lambda x: 1 if int(x) & 0x02 else 0)
df['TCP_ACK'] = df['TCP_FLAGS'].apply(lambda x: 1 if int(x) & 0x10 else 0)
df['TCP_FIN'] = df['TCP_FLAGS'].apply(lambda x: 1 if int(x) & 0x01 else 0)

# Binary features for common L7 protocols
df['IS_DNS']   = (df['L7_PROTO'] == 53).astype(int)
df['IS_HTTP']  = (df['L7_PROTO'] == 80).astype(int)
df['IS_HTTPS'] = (df['L7_PROTO'] == 443).astype(int)

# Interaction Features
df['IN_OUT_BYTES_RATIO'] = df['IN_BYTES'] / (df['OUT_BYTES'] + 1e-6)
df['IN_OUT_PKTS_RATIO']  = df['IN_PKTS']  / (df['OUT_PKTS']  + 1e-6)

# Label encoding
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Label'] = le.fit_transform(df['Label'])

# Final feature list
exclude = ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'Label', 'Attack']
feature_cols = [c for c in df.columns if c not in exclude]

print("Number of features:", len(feature_cols))
print(feature_cols)

Number of features: 56
['L4_SRC_PORT', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT', 'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT', 'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE', 'DNS_TTL_ANSWER', 'FTP_COMMAND_RET_CODE', 'IN_BYTE_PER_PKT', 'OUT_BYTE_PER_PKT', 'IN_PKT_RATE', 'OUT_PKT_RATE', 'TOTAL_PKTS', 'TOTAL_BYTES', 'FLOW_DUR_PER_PKT', 'TCP_SYN', 'TCP_ACK', 'TCP_FIN', 'I

In [3]:
print("Shape:", df.shape)
print(df['Attack'].value_counts())

Shape: (2390275, 60)
Attack
Benign            2295222
Exploits            31551
Fuzzers             22310
Generic             16560
Reconnaissance      12779
DoS                  5794
Analysis             2299
Backdoor             2169
Shellcode            1427
Worms                 164
Name: count, dtype: int64


In [4]:
#zero-days for NF-UNSW-NB15: DoS and Backdoor
zero_day_attacks = ['Dos', 'Backdoor']          
df = df.reset_index(drop=True)
zero_day_df = df[df['Attack'].str.lower().isin([a.lower() for a in zero_day_attacks])].copy()
normal_df   = df[~df['Attack'].str.lower().isin([a.lower() for a in zero_day_attacks])].copy()

cutoff1 = int(0.80 * len(normal_df))
dev_df      = normal_df.iloc[:cutoff1].copy()
test_normal = normal_df.iloc[cutoff1:].copy()

cutoff2 = int(0.90 * len(dev_df))
train_df = dev_df.iloc[:cutoff2].copy()
val_df   = dev_df.iloc[cutoff2:].copy()

print("Train     :", train_df.shape)
print("Validation:", val_df.shape)
print("Known Test:", test_normal.shape)
print("Zero-day  :", zero_day_df.shape)
print("Zero-day in train/val (must be 0):",
      train_df[train_df['Attack'].str.lower().isin([a.lower() for a in zero_day_attacks])].shape[0],
      val_df[val_df['Attack'].str.lower().isin([a.lower() for a in zero_day_attacks])].shape[0])

Train     : (1715264, 60)
Validation: (190585, 60)
Known Test: (476463, 60)
Zero-day  : (7963, 60)
Zero-day in train/val (must be 0): 0 0


In [5]:
def create_sequences(data, feature_cols, window_size=10, stride=5):
    sequences, labels, attacks = [], [], []
    feats = data[feature_cols].values.astype(np.float32)
    labs  = data['Label'].values
    atks  = data['Attack'].values
    for i in range(0, len(feats) - window_size + 1, stride):
        sequences.append(feats[i:i+window_size])
        labels.append(labs[i + window_size - 1])
        attacks.append(atks[i + window_size - 1])
    return np.array(sequences, dtype=np.float32), np.array(labels), np.array(attacks)

WINDOW, STRIDE = 10, 5

X_train, y_train, _ = create_sequences(train_df, feature_cols, WINDOW, STRIDE)
X_val,   y_val,   _ = create_sequences(val_df,   feature_cols, WINDOW, STRIDE)
X_test_known, y_test_known, atk_known = create_sequences(test_normal, feature_cols, WINDOW, STRIDE)

# Separate zero-day families
dos_df      = zero_day_df[zero_day_df['Attack'].str.lower() == 'dos']
backdoor_df = zero_day_df[zero_day_df['Attack'].str.lower() == 'backdoor']

X_dos,  y_dos,  atk_dos  = create_sequences(dos_df, feature_cols, WINDOW, STRIDE)
X_back, y_back, atk_back = create_sequences(backdoor_df, feature_cols, WINDOW, STRIDE)

X_test = np.concatenate([X_test_known, X_dos, X_back], axis=0)
y_test = np.concatenate([y_test_known, y_dos, y_back], axis=0)
attack_test = np.concatenate([atk_known, atk_dos, atk_back], axis=0)

# Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(X_train.shape)
X_val   = scaler.transform(X_val.reshape(-1, X_val.shape[-1])).reshape(X_val.shape)
X_test  = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)

# Oversampling
n_s, t, f = X_train.shape
X_train_2d, y_train = RandomOverSampler(random_state=42).fit_resample(X_train.reshape(n_s, -1), y_train)
X_train = X_train_2d.reshape(-1, t, f)

n_features = X_train.shape[-1]
print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)
print("n_features =", n_features)

Train: (668332, 10, 56) | Val: (38116, 10, 56) | Test: (96880, 10, 56)
n_features = 56


In [6]:
class StandardESN(Layer):
    def __init__(self, units=64, spectral_radius=0.9, leak_rate=0.3, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.spectral_radius = spectral_radius
        self.leak_rate = leak_rate
    def build(self, input_shape):
        feature_dim = int(input_shape[-1])
        self.W_in = self.add_weight(shape=(feature_dim, self.units), initializer='glorot_uniform', trainable=False)
        W = np.random.uniform(-1,1,(self.units,self.units)).astype(np.float32)
        W *= (np.random.rand(*W.shape) < 0.1).astype(np.float32)
        rho = np.max(np.abs(np.linalg.eigvals(W)))
        W *= (self.spectral_radius / (rho + 1e-8))
        self.W = self.add_weight(shape=(self.units,self.units), initializer=tf.constant_initializer(W), trainable=False)
        self.bias = self.add_weight(shape=(self.units,), initializer='zeros', trainable=False)
        super().build(input_shape)
    def call(self, inputs):
        batch = tf.shape(inputs)[0]
        state = tf.zeros((batch, self.units))
        def body(t, state):
            pre = tf.matmul(inputs[:,t,:], self.W_in) + tf.matmul(state, self.W) + self.bias
            return t+1, (1-self.leak_rate)*state + self.leak_rate*tf.tanh(pre)
        _, final = tf.while_loop(lambda t,s: t < tf.shape(inputs)[1], body, (0, state))
        return final

class HarmonicESN(Layer):
    def __init__(self, units=64, spectral_radius=0.9, leak_rate=0.3, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.spectral_radius = spectral_radius
        self.leak_rate = leak_rate
    def build(self, input_shape):
        feature_dim = int(input_shape[-1])
        self.W_in = self.add_weight(shape=(feature_dim, self.units), initializer='glorot_uniform', trainable=False)
        W = np.random.uniform(-1,1,(self.units,self.units)).astype(np.float32)
        W *= (np.random.rand(*W.shape) < 0.1).astype(np.float32)
        rho = np.max(np.abs(np.linalg.eigvals(W)))
        W *= (self.spectral_radius / (rho + 1e-8))
        self.W = self.add_weight(shape=(self.units,self.units), initializer=tf.constant_initializer(W), trainable=False)
        self.bias = self.add_weight(shape=(self.units,), initializer='zeros', trainable=False)
        super().build(input_shape)
    def call(self, inputs):
        batch = tf.shape(inputs)[0]
        state = tf.zeros((batch, self.units))
        def body(t, state):
            pre = tf.matmul(inputs[:,t,:], self.W_in) + tf.matmul(state, self.W) + self.bias
            return t+1, (1-self.leak_rate)*state + self.leak_rate*tf.sin(pre)
        _, final = tf.while_loop(lambda t,s: t < tf.shape(inputs)[1], body, (0, state))
        return final

class FB_HESN_Layer(Layer):
    def __init__(self, units=64, spectral_radius=0.9, leak_rate=0.3, omega_min=0.3, omega_max=3.5, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.spectral_radius = spectral_radius
        self.leak_rate = leak_rate
        self.omega_min = omega_min
        self.omega_max = omega_max
    def build(self, input_shape):
        feature_dim = int(input_shape[-1])
        self.W_in = self.add_weight(shape=(feature_dim, self.units), initializer='glorot_uniform', trainable=False)
        W = np.random.uniform(-1,1,(self.units,self.units)).astype(np.float32)
        W *= (np.random.rand(*W.shape) < 0.1).astype(np.float32)
        rho = np.max(np.abs(np.linalg.eigvals(W)))
        W *= (self.spectral_radius / (rho + 1e-8))
        self.W = self.add_weight(shape=(self.units,self.units), initializer=tf.constant_initializer(W), trainable=False)
        self.bias = self.add_weight(shape=(self.units,), initializer='zeros', trainable=False)
        omega = np.random.uniform(self.omega_min, self.omega_max, self.units).astype(np.float32)
        self.omega = self.add_weight(shape=(self.units,), initializer=tf.constant_initializer(omega), trainable=False)
        super().build(input_shape)
    def call(self, inputs):
        batch = tf.shape(inputs)[0]
        state = tf.zeros((batch, self.units))
        def body(t, state):
            pre = tf.matmul(inputs[:,t,:], self.W_in) + tf.matmul(state, self.W) + self.bias
            return t+1, (1-self.leak_rate)*state + self.leak_rate*tf.sin(self.omega * pre)
        _, final = tf.while_loop(lambda t,s: t < tf.shape(inputs)[1], body, (0, state))
        return final

def evaluate_and_store(y_true, y_pred, y_prob, attack_test, model, model_name, results_list, is_sklearn=False):
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred, average='macro')
    mcc  = matthews_corrcoef(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)
    roc = roc_auc_score(y_true, y_prob)
    pr  = average_precision_score(y_true, y_prob)

    zd = {}
    for atk in ['scanning', 'dos', 'backdoor', 'ransomware']:
        mask = np.array([str(a).lower() == atk for a in attack_test])
        total = mask.sum()
        detected = (y_pred[mask] == 1).sum() if total > 0 else 0
        zd[atk] = (detected, total, detected/total if total > 0 else np.nan)

    zd_macro = np.nanmean([zd[a][2] for a in zd])
    zd_min   = np.nanmin([zd[a][2] for a in zd])

    # Parameter count
    if is_sklearn:
        total_params = 0
        trainable = 0
        infer_time = 0.0
    else:
        total_params = model.count_params()
        trainable = int(np.sum([np.prod(v.shape) for v in model.trainable_weights]))
        # Inference time
        _ = model.predict(X_test[:128], verbose=0)
        t0 = time.time()
        _ = model.predict(X_test, batch_size=256, verbose=0)
        infer_time = time.time() - t0

    results_list.append({
        'Model': model_name,
        'Accuracy': round(acc, 4),
        'Macro-F1': round(f1, 4),
        'MCC': round(mcc, 4),
        'FPR': round(fpr, 4),
        'FNR': round(fnr, 4),
        'ROC-AUC': round(roc, 4),
        'PR-AUC': round(pr, 4),
        'Scanning': f"{zd['scanning'][0]}/{zd['scanning'][1]} ({zd['scanning'][2]:.4f})",
        'DoS': f"{zd['dos'][0]}/{zd['dos'][1]} ({zd['dos'][2]:.4f})",
        'Backdoor': f"{zd['backdoor'][0]}/{zd['backdoor'][1]} ({zd['backdoor'][2]:.4f})",
        'Ransomware': f"{zd['ransomware'][0]}/{zd['ransomware'][1]} ({zd['ransomware'][2]:.4f})",
        'ZD-MacroRecall': round(zd_macro, 4),
        'ZD-MinRecall': round(zd_min, 4),
        'Total Params': total_params,
        'Trainable Params': trainable,
        'Inference Time (s)': round(infer_time, 2)
    })
    print(f"{model_name} done | Macro-F1={f1:.4f} | ZD-Macro={zd_macro:.4f}")

In [7]:
results = []

def train_neural(build_fn, name):
    tf.keras.backend.clear_session()
    set_seed(42)
    model = build_fn()
    model.compile(optimizer=Adam(0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=40, batch_size=256,
              callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)],
              verbose=1)
    y_prob = model.predict(X_test, batch_size=256, verbose=0)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    evaluate_and_store(y_test, y_pred, y_prob, attack_test, model, name, results)

# Build functions (units=64)
def build_standard_esn():
    inputs = Input(shape=(10, n_features))
    x = StandardESN(units=64)(inputs)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.3)(x)
    return Model(inputs, Dense(2, activation='softmax')(x))

def build_harmonic_esn():
    inputs = Input(shape=(10, n_features))
    x = HarmonicESN(units=64)(inputs)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.3)(x)
    return Model(inputs, Dense(2, activation='softmax')(x))

def build_fb_hesn():
    inputs = Input(shape=(10, n_features))
    x = FB_HESN_Layer(units=64, spectral_radius=0.9, leak_rate=0.3,
                      omega_min=0.3, omega_max=1.0)(inputs)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.3)(x)
    return Model(inputs, Dense(2, activation='softmax')(x))

def build_gru():
    inputs = Input(shape=(10, n_features))
    x = GRU(64)(inputs)
    x = Dropout(0.3)(x)
    x = Dense(32, activation='relu')(x)
    return Model(inputs, Dense(2, activation='softmax')(x))

# Run the important models
train_neural(build_standard_esn, "Standard ESN-64")
train_neural(build_harmonic_esn, "Harmonic ESN-64")
train_neural(build_fb_hesn, "FB-HESN-64")
train_neural(build_gru, "GRU-64")

df_master = pd.DataFrame(results)
print(df_master.to_string(index=False))
df_master.to_csv("UNSW_master_64.csv", index=False)


Epoch 1/40
2611/2611 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.9645 - loss: 0.1113 - val_accuracy: 0.9593 - val_loss: 0.1290
Epoch 2/40
2611/2611 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.9858 - loss: 0.0545 - val_accuracy: 0.9691 - val_loss: 0.1089
Epoch 3/40
2611/2611 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.9882 - loss: 0.0464 - val_accuracy: 0.9721 - val_loss: 0.1025
Epoch 4/40
2611/2611 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - accuracy: 0.9894 - loss: 0.0419 - val_accuracy: 0.9747 - val_loss: 0.0935
Epoch 5/40
2611/2611 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.9901 - loss: 0.0391 - val_accuracy: 0.9743 - val_loss: 0.0922
Epoch 6/40
2611/2611 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step - accuracy: 0.9905 - loss: 0.0371 - val_accuracy: 0.9760 - val_loss: 0.0865
Epoch 7/40
2611/2611 ━━━━━━━━━━━━━━━━━━━━ 16s 6ms/step - accuracy: 0.9908 - loss: 0.0358 - val_accuracy: 0.9771 - val_loss: 0.0812
Epoch 8/40
2611/2611 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.9910 - loss: 